#Gold Layer ETL Pipeline

Digital Lifestyle Benchmark Dataset — Medallion Architecture

This notebook implements the Gold Layer for the Digital Lifestyle Benchmark dataset.

The Silver dataset is transformed into a dimensional model using a Star Schema design. Dimension tables are created to store descriptive business attributes, while a central Fact table stores measurable metrics.

The final outputs are:
*   Gold_Layer/Dim_Person.csv
*   Gold_Layer/Dim_Device.csv
*   Gold_Layer/Dim_Health_Profile.csv
*   Gold_Layer/Dim_Geography.csv
*   Gold_Layer/Fact_Digital_Lifestyle.csv

These tables are optimized for Power BI reporting and analytical workloads.

##1. Define File Paths & Create Gold Layer Folder

In [ ]:
SILVER_FILE_PATH = "Silver_Layer/Digital_Lifestyle_Benchmark_Silver.csv"

GOLD_FOLDER = "Gold_Layer"

os.makedirs(GOLD_FOLDER, exist_ok=True)
print(f"Gold Layer folder ready at: ./{GOLD_FOLDER}/")


def gold_path(filename: str) -> str:
    """Helper — returns full path for a Gold Layer output file."""
    return os.path.join(GOLD_FOLDER, filename)

Gold Layer folder ready at: ./Gold_Layer/


##2. Load Silver Dataset

In [ ]:
df = pd.read_csv(SILVER_FILE_PATH)

print("Silver dataset loaded successfully.")
print(f"Shape: {df.shape}")

Silver dataset loaded successfully.
Shape: (100000, 30)


##3. Data Quality Validation

In [ ]:
print("===== DATA VALIDATION =====")

print("\nMissing Values:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "None — dataset is clean.")

print("\nDuplicate IDs:")
print(df["id"].duplicated().sum())

print("\nDataset Shape:")
print(df.shape)

===== DATA VALIDATION =====

Missing Values:
None — dataset is clean.

Duplicate IDs:
0

Dataset Shape:
(100000, 30)


##4. Build dim_person
__________________________________
 Captures the unique demographic combinations present in the
 Silver data. One row per unique (gender, age_group,
 income_level, education_level, daily_role) combination.

In [ ]:
PERSON_COLS = [
    "gender",
    "age_group",
    "income_level",
    "education_level",
    "daily_role",
]

dim_person = (
    df[PERSON_COLS]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Assign surrogate key (1-based).
dim_person.insert(0, "person_key", range(1, len(dim_person) + 1))

# Merge person_key back into the main DataFrame as FK.
df = df.merge(dim_person, on=PERSON_COLS, how="left")

# Save dimension table.
dim_person.to_csv(gold_path("dim_person.csv"), index=False)

print(f"\ndim_person built        : {dim_person.shape[0]} rows, {dim_person.shape[1]} columns")
print(f"Unique person profiles  : {dim_person.shape[0]}")


dim_person built        : 574 rows, 6 columns
Unique person profiles  : 574





## 5. Build dim_device
 Captures the unique device & screen-time category combinations.

In [ ]:
DEVICE_COLS = [
    "device_type",
    "screen_time_category",
]

dim_device = (
    df[DEVICE_COLS]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_device.insert(0, "device_key", range(1, len(dim_device) + 1))

df = df.merge(dim_device, on=DEVICE_COLS, how="left")

dim_device.to_csv(gold_path("dim_device.csv"), index=False)

print(f"\ndim_device built        : {dim_device.shape[0]} rows, {dim_device.shape[1]} columns")


dim_device built        : 16 rows, 3 columns


## 6. Build dim_health_profile
 Captures the unique combinations of categorical health &
 productivity labels derived in the Silver Layer.

In [ ]:
HEALTH_COLS = [
    "sleep_category",
    "mental_health_risk",
    "productivity_category",
]

dim_health_profile = (
    df[HEALTH_COLS]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_health_profile.insert(0, "health_key", range(1, len(dim_health_profile) + 1))

df = df.merge(dim_health_profile, on=HEALTH_COLS, how="left")

dim_health_profile.to_csv(gold_path("dim_health_profile.csv"), index=False)

print(f"\ndim_health_profile built: {dim_health_profile.shape[0]} rows, {dim_health_profile.shape[1]} columns")


dim_health_profile built: 47 rows, 4 columns


#7. Build dim_geography ───────────────────────────────────
 Simple region lookup — one row per distinct region value.

In [ ]:
GEO_COLS = ["region"]

dim_geography = (
    df[GEO_COLS]
    .drop_duplicates()
    .reset_index(drop=True)
    .sort_values("region")
    .reset_index(drop=True)
)

dim_geography.insert(0, "geo_key", range(1, len(dim_geography) + 1))

df = df.merge(dim_geography, on=GEO_COLS, how="left")

dim_geography.to_csv(gold_path("dim_geography.csv"), index=False)

print(f"\ndim_geography built     : {dim_geography.shape[0]} rows, {dim_geography.shape[1]} columns")


dim_geography built     : 6 rows, 2 columns


#8. Validate FK Joins ─────────────────────────────────────
 Every FK column must have zero NaN values after the merges.
 A NaN here means a Silver row had a value not covered by its
 dimension table — which would indicate a join bug.

In [ ]:
fk_cols = ["person_key", "device_key", "health_key", "geo_key"]

print("\n===== FK JOIN VALIDATION =====")
all_ok = True
for fk in fk_cols:
    nulls = df[fk].isnull().sum()
    status = "✓ OK" if nulls == 0 else f"✗ {nulls} NULLs — JOIN ERROR"
    print(f"  {fk:<15}: {status}")
    if nulls > 0:
        all_ok = False

if not all_ok:
    raise ValueError(
        "One or more FK columns contain NULLs. "
        "Check dimension build steps above."
    )

print("All FK joins validated successfully.")
print("==============================")



===== FK JOIN VALIDATION =====
  person_key     : ✓ OK
  device_key     : ✓ OK
  health_key     : ✓ OK
  geo_key        : ✓ OK
All FK joins validated successfully.


##9. Build fact_digital_lifestyle

In [ ]:
MEASURE_COLS = [
    "age",
    "device_hours_per_day",
    "phone_unlocks",
    "notifications_per_day",
    "social_media_mins",
    "study_mins",
    "physical_activity_days",
    "sleep_hours",
    "sleep_quality",
    "anxiety_score",
    "depression_score",
    "stress_level",
    "happiness_score",
    "focus_score",
    "productivity_score",
    "digital_dependence_score",
    "digital_wellness_score",
    "high_risk_flag",
]

fact_digital_lifestyle = df[
    fk_cols + ["id"] + MEASURE_COLS
].copy()

# Rename id to original_id for clarity inside the fact table.
fact_digital_lifestyle = fact_digital_lifestyle.rename(
    columns={"id": "original_id"}
)

# Assign surrogate fact key.
fact_digital_lifestyle.insert(
    0,
    "fact_id",
    range(1, len(fact_digital_lifestyle) + 1),
)

fact_digital_lifestyle.to_csv(
    gold_path("fact_digital_lifestyle.csv"), index=False
)

print(f"\nfact_digital_lifestyle  : {fact_digital_lifestyle.shape[0]} rows, {fact_digital_lifestyle.shape[1]} columns")
print(f"FK columns              : {fk_cols}")
print(f"Measure columns         : {len(MEASURE_COLS)}")


fact_digital_lifestyle  : 100000 rows, 24 columns
FK columns              : ['person_key', 'device_key', 'health_key', 'geo_key']
Measure columns         : 18


##11. Verify Star Schema Integrity  
 Confirm every FK value in the fact table exists as a PK in its
 dimension — this is the referential integrity check.

In [ ]:
print("\n===== STAR SCHEMA INTEGRITY =====")

integrity_checks = [
    ("person_key", dim_person,         "person_key"),
    ("device_key", dim_device,         "device_key"),
    ("health_key", dim_health_profile, "health_key"),
    ("geo_key",    dim_geography,      "geo_key"),
]

for fk_col, dim_df, pk_col in integrity_checks:
    fact_keys = set(fact_digital_lifestyle[fk_col].unique())
    dim_keys  = set(dim_df[pk_col].unique())
    orphans   = fact_keys - dim_keys
    status    = "✓ OK" if not orphans else f"✗ {len(orphans)} orphan keys"
    print(f"  fact → {dim_df.columns[0]:<20}: {status}")


===== STAR SCHEMA INTEGRITY =====
  fact → person_key          : ✓ OK
  fact → device_key          : ✓ OK
  fact → health_key          : ✓ OK
  fact → geo_key             : ✓ OK


#Summary & Next Steps

The Gold Layer is complete:

* Star Schema implemented.
* Dimension tables created.
* Surrogate keys generated.
* Fact table created.
* Foreign key relationships established.
* Data model optimized for Power BI.

Next stage:

Power BI Semantic Model

Relationships:

Dim_Person[person_key]
→ Fact_Digital_Lifestyle[person_key]

Dim_Device[device_key]
→ Fact_Digital_Lifestyle[device_key]

Dim_Health_Profile[health_key]
→ Fact_Digital_Lifestyle[health_key]

Dim_Geography[geo_key]
→ Fact_Digital_Lifestyle[geo_key]

Relationship Type:

One-to-Many

Cross Filter Direction:

Single